# Project 2 — Your First Real Metrics

You have ~2,500 days x 10 stocks of returns sitting in `data/`. On their own, that's
an unreadable wall of numbers. This notebook collapses all of it into a handful of
numbers that actually mean something, for each stock:

- **Annualised return (CAGR)** — how much it grew per year, on average.
- **Annualised volatility** — how violently it swung around. The basic measure of *risk*.
- **Sharpe ratio** — return *per unit of risk*. The number a quant actually cares about.
- **Max drawdown** — the worst peak-to-trough fall. "If I'd bought at the worst moment, how far down would I have been?"

Then we'll rank the ten stocks and chart them — and you'll see that the biggest
headline return usually isn't the best investment once risk is in the picture.

> Run each cell top to bottom with **Shift+Enter**. Make sure the kernel (top right)
> says **quant**.

## Step 1 — Load the clean data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# index_col=0 -> the date column becomes the row label
# parse_dates=True -> treat those labels as real dates
prices  = pd.read_csv("../data/prices.csv",  index_col=0, parse_dates=True)
returns = pd.read_csv("../data/returns.csv", index_col=0, parse_dates=True)

returns = returns.dropna(how="all")   # first row has no "day before", so it's empty

print("Prices: ", prices.shape[0], "days x", prices.shape[1], "stocks")
prices.tail()

## Step 2 — The numbers

A few conventions, so nothing here is magic:

- **252** is the number of trading days in a year (markets are shut on weekends/holidays).
  To turn a *daily* number into a *yearly* one, we scale by 252 — and because volatility
  grows with the square root of time, we scale risk by `sqrt(252)`.
- **Sharpe ratio** = annual return per unit of annual risk. We assume a risk-free rate of
  0 here to keep it simple (in reality you'd subtract the cash/bond rate). Higher = more
  return for the risk taken. As a rough feel: under 1 is mediocre, around 1 is decent,
  above 2 is excellent (and rare).


In [ ]:
TRADING_DAYS = 252

# --- Annualised return (CAGR): geometric growth per year, straight from prices ---
def cagr(col):
    col = col.dropna()
    years = len(col) / TRADING_DAYS
    return (col.iloc[-1] / col.iloc[0]) ** (1 / years) - 1

ann_return = prices.apply(cagr)

# --- Annualised volatility: daily std of returns, scaled up to a year ---
ann_vol = returns.std() * np.sqrt(TRADING_DAYS)

# --- Sharpe ratio: annualised mean return / annualised volatility (risk-free = 0) ---
sharpe = (returns.mean() * TRADING_DAYS) / ann_vol

# Bundle into one table
summary = pd.DataFrame({
    "Ann. Return": ann_return,
    "Ann. Volatility": ann_vol,
    "Sharpe": sharpe,
})
summary

## Step 3 — Max drawdown (the number that tells you what you could *stomach*)

In [ ]:
# Drawdown = how far below the previous peak the price is, at every point in time.
# cummax() = the highest price seen SO FAR (the running peak).
# price / peak - 1  -> 0 when at a new high, negative when below it.
running_peak = prices.cummax()
drawdown = prices / running_peak - 1

# The worst (most negative) drawdown each stock ever suffered:
max_dd = drawdown.min()

summary["Max Drawdown"] = max_dd
summary

## Step 4 — Rank them honestly

Now sort by **Sharpe** — return per unit of risk — instead of by raw return.
Watch what moves up and what falls down the list.

In [ ]:
ranked = summary.sort_values("Sharpe", ascending=False)

# Show as nice percentages (Sharpe stays a plain number)
styled = ranked.copy()
for c in ["Ann. Return", "Ann. Volatility", "Max Drawdown"]:
    styled[c] = (styled[c] * 100).round(1).astype(str) + "%"
styled["Sharpe"] = styled["Sharpe"].round(2)
styled

**Look at this table for a moment before moving on.**

- Is the stock with the highest *return* also the one with the best *Sharpe*? Usually not —
  high returns often come strapped to high volatility, and Sharpe is what cuts through that.
- Look at the **Max Drawdown** column. A stock that lost 50%+ at its worst point would have
  tested your nerve hard, even if it ended up fine. That column is the "could I actually have
  held this?" check.

This single table already puts you ahead of most people picking stocks off vibes.

## Step 5 — Chart 1: equity curves (what $100 became)

In [ ]:
# Normalise each stock to start at 100, so they're comparable on one chart
# regardless of their actual share price.
first_value = prices.apply(lambda c: c.loc[c.first_valid_index()])
equity = prices / first_value * 100

plt.figure(figsize=(12, 6))
for col in equity.columns:
    plt.plot(equity.index, equity[col], label=col, linewidth=1)
plt.title("Growth of $100 invested (each stock)")
plt.ylabel("Value ($, started at 100)")
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.3)
plt.show()

The steepest line made the most money — but notice how *bumpy* some of the steep ones
are. Bumpiness is volatility, and volatility is risk. The next chart makes the pain explicit.

## Step 6 — Chart 2: drawdowns (the pain, visualised)

In [ ]:
plt.figure(figsize=(12, 6))
for col in drawdown.columns:
    plt.plot(drawdown.index, drawdown[col] * 100, label=col, linewidth=1)
plt.title("Drawdown over time (how far below the previous peak)")
plt.ylabel("Drawdown (%)")
plt.legend(ncol=2, fontsize=8)
plt.grid(alpha=0.3)
plt.show()

Every dip below 0 is a period where you'd have been holding at a loss versus the peak.
The deep, long troughs are the stretches that make people panic-sell at the bottom — which
is exactly when they shouldn't. Seeing this is *why* max drawdown matters as much as return.

---

### What you just learned
1. How to turn raw price data into risk-adjusted performance numbers.
2. That **return without risk context is meaningless** — Sharpe is the honest comparison.
3. That **drawdown is the human side of risk** — the number that decides whether you could
   actually stick with something.

### Try it yourself
- Re-sort the table by `Ann. Return` instead of `Sharpe`. Does the order change? That *is*
  the lesson.
- Which stock would you rather have held: the highest return, or the best Sharpe? Why?